# Inspect Final Project Tables From `BOW` Downward

This notebook documents each unique UVA Box table referenced in `FinalProject.ipynb` from the `BOW` section downward. Each table gets its own pair of cells:

- a short markdown description of the table being inspected
- a code cell that loads the table and reports its shape, number of observations, column names, and `head(10)`

The loader tries the UVA Box link first and falls back to the matching local project file when needed.

In [1]:
from io import BytesIO
from pathlib import Path
from textwrap import fill
from urllib.parse import parse_qsl, urlencode, urlparse, urlunparse

import pandas as pd
import requests
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)

PROJECT_ROOT = Path.cwd()

def add_download_flag(url: str) -> str:
    parsed = urlparse(url)
    query = dict(parse_qsl(parsed.query, keep_blank_values=True))
    query['download'] = '1'
    return urlunparse(parsed._replace(query=urlencode(query)))


def load_table(box_url: str, local_path: str, file_format: str) -> tuple[pd.DataFrame, str]:
    local_file = PROJECT_ROOT / local_path
    try:
        response = requests.get(add_download_flag(box_url), timeout=90)
        response.raise_for_status()
        if file_format == 'parquet':
            df = pd.read_parquet(BytesIO(response.content))
        elif file_format == 'csv':
            df = pd.read_csv(BytesIO(response.content))
        else:
            raise ValueError(f'Unsupported format: {file_format}')
        return df, 'Box download'
    except Exception:
        if not local_file.exists():
            raise FileNotFoundError(f'Could not load {local_path} from Box or local workspace.')
        if file_format == 'parquet':
            df = pd.read_parquet(local_file)
        elif file_format == 'csv':
            df = pd.read_csv(local_file)
        else:
            raise ValueError(f'Unsupported format: {file_format}')
        return df, 'Local fallback'


def inspect_table(section: str, box_url: str, local_path: str, file_format: str, preview_rows: int = 10) -> pd.DataFrame:
    df, load_source = load_table(box_url, local_path, file_format)
    print(f'Section: {section}')
    print(f'Box URL: {box_url}')
    print(f'Local file: {local_path}')
    print(f'Loaded from: {load_source}')
    print(f'Format: {file_format}')
    print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    print(f'Number of observations: {df.shape[0]:,}')
    print('Column names:')
    print(fill(', '.join(str(col) for col in df.columns), width=100))
    display(df.head(preview_rows))
    return df

## BOW

This section inspects the project `BOW` table, the long bag-of-words table used as the starting point for several downstream models.

The saved `bow.parquet` now includes both the raw term count column `n` and a row-level `tfidf` value for each observed term occurrence in each bag.

In [2]:
BOW = inspect_table(
    section='BOW',
    box_url='https://virginia.box.com/shared/static/hyvirbfhuo5vcjegyjhsrn2ea1d6ikhq.parquet',
    local_path='bow.parquet',
    file_format='parquet'
)

Section: BOW
Box URL: https://virginia.box.com/shared/static/hyvirbfhuo5vcjegyjhsrn2ea1d6ikhq.parquet
Local file: bow.parquet
Loaded from: Box download
Format: parquet
Shape: 1,775,569 rows x 5 columns
Number of observations: 1,775,569
Column names:
country_id, article_n, term_str, n, tfidf


,country_id,article_n,term_str,n,tfidf
0,Afghanistan,1,accordance,1,2.950516
1,Afghanistan,1,adhering,1,9.123469
2,Afghanistan,1,admiring,1,10.732907
3,Afghanistan,1,afghanistan,3,22.602701
4,Afghanistan,1,all,3,8.100179
5,Afghanistan,1,allah,2,15.283729
6,Afghanistan,1,almighty,1,6.597740
7,Afghanistan,1,an,1,2.501132
8,Afghanistan,1,and,18,21.972341
9,Afghanistan,1,appreciating,1,9.816616


## DTM

This section inspects the document-term matrix derived from the project `BOW` table.

In [3]:
DTM = inspect_table(
    section='DTM',
    box_url='https://virginia.box.com/shared/static/tipo8kaykn4xapb85etbvtiq8b5w07ut',
    local_path='dtm.parquet',
    file_format='parquet'
)

Section: DTM
Box URL: https://virginia.box.com/shared/static/tipo8kaykn4xapb85etbvtiq8b5w07ut
Local file: dtm.parquet
Loaded from: Box download
Format: parquet
Shape: 33,726 rows x 24,735 columns
Number of observations: 33,726
Column names:
aa, aalim, aargau, ab, aba, ababa, abadam, abaiang, abaji, abak, abakaliki, abandon, abandoned,
abandoning, abandonment, abandons, abariringa, abases, abasi, abasto, abate, abatement, abatements,
abattoir, abattoirs, abbot, abbottabad, abbreviated, abbreviations, abd-allah, abdallah, abdel,
abdicate, abdicated, abdicates, abdication, abdications, abduct, abduction, abdul, abdulai,
abdulaziz, abdullah, abemama, abeokuta, abetment, abets, abetting, abeyance, abi, abia, abide,
abided, abides, abiding, abilities, ability, abim, abinger, abish, abjure, abkhazia, abkhazian,
able, able-bodied, abler, ablution, ably, abnegation, abnormal, abnormalities, abnormality,
abnormally, aboard, abode, aboh-mbaise, abolish, abolished, abolishes, abolishing, abolishme

term_str               aa  aalim  aargau  ab  aba  ababa  abadam  abaiang  \
country_id  article_n                                                       
Afghanistan 1           0      0       0   0    0      0       0        0   
            2           0      0       0   0    0      0       0        0   
            3           0      0       0   0    0      0       0        0   
            4           0      0       0   0    0      0       0        0   
            5           0      0       0   0    0      0       0        0   
            6           0      0       0   0    0      0       0        0   
            7           0      0       0   0    0      0       0        0   
            8           0      0       0   0    0      0       0        0   
            9           0      0       0   0    0      0       0        0   
            10          0      0       0   0    0      0       0        0   

term_str               abaji  abak  abakaliki  abandon  abandoned  abandoning  \
country_id  article_n                                                           
Afghanistan 1              0     0          0        0          0           0   
            2              0     0          0        0          0           0   
            3              0     0          0        0          0           0   
            4              0     0          0        0          0           0   
            5              0     0          0        0          0           0   
            6              0     0          0        0          0           0   
            7              0     0          0        0          0           0   
            8              0     0          0        0          0           0   
            9              0     0          0        0          0           0   
            10             0     0          0        0          0           0   

term_str               abandonment  abandons  abariringa  abases  abasi  \
country_id  article_n                                                     
Afghanistan 1                    0         0           0       0      0   
            2                    0         0           0       0      0   
            3                    0         0           0       0      0   
            4                    0         0           0       0      0   
            5                    0         0           0       0      0   
            6                    0         0           0       0      0   
            7                    0         0           0       0      0   
            8                    0         0           0       0      0   
            9                    0         0           0       0      0   
            10                   0         0           0       0      0   

term_str               abasto  abate  abatement  abatements  abattoir  \
country_id  article_n                                                   
Afghanistan 1               0      0          0           0         0   
            2               0      0          0           0         0   
            3               0      0          0           0         0   
            4               0      0          0           0         0   
            5               0      0          0           0         0   
            6               0      0          0           0         0   
            7               0      0          0           0         0   
            8               0      0          0           0         0   
            9               0      0          0           0         0   
            10              0      0          0           0         0   

term_str               abattoirs  abbot  abbottabad  abbreviated  \
country_id  article_n                                              
Afghanistan 1                  0      0           0            0   
            2                  0      0           0            0   
            3                  0      0           0 

## TFIDF

This section inspects the TFIDF-weighted document-term matrix built from the project corpus.

In [4]:
TFIDF = inspect_table(
    section='TFIDF',
    box_url='https://virginia.box.com/shared/static/eypbjdf47oiftzj9vdsz7cz5cas2vp81',
    local_path='tfidf.parquet',
    file_format='parquet'
)

Section: TFIDF
Box URL: https://virginia.box.com/shared/static/eypbjdf47oiftzj9vdsz7cz5cas2vp81
Local file: tfidf.parquet
Loaded from: Box download
Format: parquet
Shape: 33,726 rows x 24,735 columns
Number of observations: 33,726
Column names:
aa, aalim, aargau, ab, aba, ababa, abadam, abaiang, abaji, abak, abakaliki, abandon, abandoned,
abandoning, abandonment, abandons, abariringa, abases, abasi, abasto, abate, abatement, abatements,
abattoir, abattoirs, abbot, abbottabad, abbreviated, abbreviations, abd-allah, abdallah, abdel,
abdicate, abdicated, abdicates, abdication, abdications, abduct, abduction, abdul, abdulai,
abdulaziz, abdullah, abemama, abeokuta, abetment, abets, abetting, abeyance, abi, abia, abide,
abided, abides, abiding, abilities, ability, abim, abinger, abish, abjure, abkhazia, abkhazian,
able, able-bodied, abler, ablution, ably, abnegation, abnormal, abnormalities, abnormality,
abnormally, aboard, abode, aboh-mbaise, abolish, abolished, abolishes, abolishing, aboli

term_str                aa  aalim  aargau   ab  aba  ababa  abadam  abaiang  \
country_id  article_n                                                         
Afghanistan 1          0.0    0.0     0.0  0.0  0.0    0.0     0.0      0.0   
            2          0.0    0.0     0.0  0.0  0.0    0.0     0.0      0.0   
            3          0.0    0.0     0.0  0.0  0.0    0.0     0.0      0.0   
            4          0.0    0.0     0.0  0.0  0.0    0.0     0.0      0.0   
            5          0.0    0.0     0.0  0.0  0.0    0.0     0.0      0.0   
            6          0.0    0.0     0.0  0.0  0.0    0.0     0.0      0.0   
            7          0.0    0.0     0.0  0.0  0.0    0.0     0.0      0.0   
            8          0.0    0.0     0.0  0.0  0.0    0.0     0.0      0.0   
            9          0.0    0.0     0.0  0.0  0.0    0.0     0.0      0.0   
            10         0.0    0.0     0.0  0.0  0.0    0.0     0.0      0.0   

term_str               abaji  abak  abakaliki  abandon  abandoned  abandoning  \
country_id  article_n                                                           
Afghanistan 1            0.0   0.0        0.0      0.0        0.0         0.0   
            2            0.0   0.0        0.0      0.0        0.0         0.0   
            3            0.0   0.0        0.0      0.0        0.0         0.0   
            4            0.0   0.0        0.0      0.0        0.0         0.0   
            5            0.0   0.0        0.0      0.0        0.0         0.0   
            6            0.0   0.0        0.0      0.0        0.0         0.0   
            7            0.0   0.0        0.0      0.0        0.0         0.0   
            8            0.0   0.0        0.0      0.0        0.0         0.0   
            9            0.0   0.0        0.0      0.0        0.0         0.0   
            10           0.0   0.0        0.0      0.0        0.0         0.0   

term_str               abandonment  abandons  abariringa  abases  abasi  \
country_id  article_n                                                     
Afghanistan 1                  0.0       0.0         0.0     0.0    0.0   
            2                  0.0       0.0         0.0     0.0    0.0   
            3                  0.0       0.0         0.0     0.0    0.0   
            4                  0.0       0.0         0.0     0.0    0.0   
            5                  0.0       0.0         0.0     0.0    0.0   
            6                  0.0       0.0         0.0     0.0    0.0   
            7                  0.0       0.0         0.0     0.0    0.0   
            8                  0.0       0.0         0.0     0.0    0.0   
            9                  0.0       0.0         0.0     0.0    0.0   
            10                 0.0       0.0         0.0     0.0    0.0   

term_str               abasto  abate  abatement  abatements  abattoir  \
country_id  article_n                                                   
Afghanistan 1             0.0    0.0        0.0         0.0       0.0   
            2             0.0    0.0        0.0         0.0       0.0   
            3             0.0    0.0        0.0         0.0       0.0   
            4             0.0    0.0        0.0         0.0       0.0   
            5             0.0    0.0        0.0         0.0       0.0   
            6             0.0    0.0        0.0         0.0       0.0   
            7             0.0    0.0        0.0         0.0       0.0   
            8             0.0    0.0        0.0         0.0       0.0   
            9             0.0    0.0        0.0         0.0       0.0   
            10            0.0    0.0        0.0         0.0       0.0   

term_str               abattoirs  abbot  abbottabad  abbreviated  \
country_id  article_n                                              
Afghanistan 1                0.0    0.0         0.0          0.0   
            2                0.0    0.0         0.0          0.0   
            3               

## TFIDF_REDUCED_L2

This section inspects the reduced and L2-normalized TFIDF table used for downstream modeling.

In [5]:
TFIDF_REDUCED_L2 = inspect_table(
    section='TFIDF_REDUCED_L2',
    box_url='https://virginia.box.com/shared/static/0c2vu4l7moiq1jjxdcy1v3aeoyiu0uur',
    local_path='tfidf_reduced_l2.parquet',
    file_format='parquet'
)

Section: TFIDF_REDUCED_L2
Box URL: https://virginia.box.com/shared/static/0c2vu4l7moiq1jjxdcy1v3aeoyiu0uur
Local file: tfidf_reduced_l2.parquet
Loaded from: Box download
Format: parquet
Shape: 33,726 rows x 3,006 columns
Number of observations: 33,726
Column names:
abide, ability, able, abolish, abolished, abolition, about, above, abroad, abrogated, absence,
absent, absolute, abuse, academic, accept, acceptance, accepted, accepts, access, accessible,
accompanied, accord, accorded, according, accordingly, account, accountability, accountable,
accounting, accounts, accredited, accusation, accused, achieve, achieving, acquire, acquired,
acquisition, acquitted, acted, acting, action, actions, active, activities, activity, acts, actual,
ad, adaptations, added, addition, additional, address, addressed, adequate, adherence, adjourn,
adjournment, adjudged, adjudicate, administer, administered, administering, administration,
administrations, administrative, administrator, admissible, admission,

term_str               abide  ability  able  abolish  abolished  abolition  \
country_id  article_n                                                        
Afghanistan 1            0.0      0.0   0.0      0.0        0.0        0.0   
            2            0.0      0.0   0.0      0.0        0.0        0.0   
            3            0.0      0.0   0.0      0.0        0.0        0.0   
            4            0.0      0.0   0.0      0.0        0.0        0.0   
            5            0.0      0.0   0.0      0.0        0.0        0.0   
            6            0.0      0.0   0.0      0.0        0.0        0.0   
            7            0.0      0.0   0.0      0.0        0.0        0.0   
            8            0.0      0.0   0.0      0.0        0.0        0.0   
            9            0.0      0.0   0.0      0.0        0.0        0.0   
            10           0.0      0.0   0.0      0.0        0.0        0.0   

term_str               about  above  abroad  abrogated  absence  absent  \
country_id  article_n                                                     
Afghanistan 1            0.0    0.0     0.0        0.0      0.0     0.0   
            2            0.0    0.0     0.0        0.0      0.0     0.0   
            3            0.0    0.0     0.0        0.0      0.0     0.0   
            4            0.0    0.0     0.0        0.0      0.0     0.0   
            5            0.0    0.0     0.0        0.0      0.0     0.0   
            6            0.0    0.0     0.0        0.0      0.0     0.0   
            7            0.0    0.0     0.0        0.0      0.0     0.0   
            8            0.0    0.0     0.0        0.0      0.0     0.0   
            9            0.0    0.0     0.0        0.0      0.0     0.0   
            10           0.0    0.0     0.0        0.0      0.0     0.0   

term_str               absolute  abuse  academic  accept  acceptance  \
country_id  article_n                                                  
Afghanistan 1               0.0    0.0       0.0     0.0         0.0   
            2               0.0    0.0       0.0     0.0         0.0   
            3               0.0    0.0       0.0     0.0         0.0   
            4               0.0    0.0       0.0     0.0         0.0   
            5               0.0    0.0       0.0     0.0         0.0   
            6               0.0    0.0       0.0     0.0         0.0   
            7               0.0    0.0       0.0     0.0         0.0   
            8               0.0    0.0       0.0     0.0         0.0   
            9               0.0    0.0       0.0     0.0         0.0   
            10              0.0    0.0       0.0     0.0         0.0   

term_str               accepted  accepts  access  accessible  accompanied  \
country_id  article_n                                                       
Afghanistan 1               0.0      0.0     0.0         0.0          0.0   
            2               0.0      0.0     0.0         0.0          0.0   
            3               0.0      0.0     0.0         0.0          0.0   
            4               0.0      0.0     0.0         0.0          0.0   
            5               0.0      0.0     0.0         0.0          0.0   
            6               0.0      0.0     0.0         0.0          0.0   
            7               0.0      0.0     0.0         0.0          0.0   
            8               0.0      0.0     0.0         0.0          0.0   
            9               0.0      0.0     0.0         0.0          0.0   
            10              0.0      0.0     0.0         0.0          0.0   

term_str               accord  accorded  according  accordingly  account  \
country_id  article_n                                                      
Afghanistan 1             0.0       0.0        0.0          0.0      0.0   
            2             0.0       0.0        0.0          0.0      0.0   
            3             0.0       0.0        0.0          0.0     

## PCA Components

This section inspects the PCA components table produced from the reduced TFIDF representation.

In [6]:
PCA_COMPS = inspect_table(
    section='PCA Components',
    box_url='https://virginia.box.com/shared/static/z8ytkodps3omi9wmgf8vdmk98s8s8c0h',
    local_path='pca_comps.parquet',
    file_format='parquet'
)

Section: PCA Components
Box URL: https://virginia.box.com/shared/static/z8ytkodps3omi9wmgf8vdmk98s8s8c0h
Local file: pca_comps.parquet
Loaded from: Box download
Format: parquet
Shape: 3,006 rows x 3,006 columns
Number of observations: 3,006
Column names:
abide, ability, able, abolish, abolished, abolition, about, above, abroad, abrogated, absence,
absent, absolute, abuse, academic, accept, acceptance, accepted, accepts, access, accessible,
accompanied, accord, accorded, according, accordingly, account, accountability, accountable,
accounting, accounts, accredited, accusation, accused, achieve, achieving, acquire, acquired,
acquisition, acquitted, acted, acting, action, actions, active, activities, activity, acts, actual,
ad, adaptations, added, addition, additional, address, addressed, adequate, adherence, adjourn,
adjournment, adjudged, adjudicate, administer, administered, administering, administration,
administrations, administrative, administrator, admissible, admission, admitted, 

term_str,abide,ability,able,abolish,abolished,abolition,about,above,abroad,abrogated,absence,absent,absolute,abuse,academic,accept,acceptance,accepted,accepts,access,accessible,accompanied,accord,accorded,according,accordingly,account,accountability,accountable,accounting,accounts,accredited,accusation,accused,achieve,achieving,acquire,acquired,acquisition,acquitted,acted,acting,action,actions,active,activities,activity,acts,actual,ad,...,well,well-being,what,whatever,whatsoever,whenever,where,whereas,whereby,whether,whichever,while,white,whoever,whole,wholly,whom,whose,wish,wishes,withdraw,withdrawal,withdrawn,witness,witnesses,woman,women,words,work,worker,workers,working,works,world,worship,would,writ,writing,writs,written,yang,year,yet,young,youth,yuan,zambia,zimbabwe,zone,zones
component_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
PC0,0.004547,0.004360,-0.003622,-0.000026,-0.001277,-0.001204,0.003576,-0.003921,0.009625,0.001503,-0.040956,-0.020681,-0.027455,0.007871,0.005867,-0.004586,-1.585838e-03,-0.002106,-0.001262,0.039567,0.005799,-0.002040,-0.000838,0.002840,0.014254,-0.004795,0.000185,0.004487,-0.000164,-0.000059,-0.006864,-0.000435,-0.002457,-0.000770,0.012532,0.005673,0.006993,0.006818,0.007413,-0.000614,-0.005042,-0.089392,0.001043,0.010758,0.005878,0.040548,0.025211,-0.000450,0.000529,-0.002677,...,0.060130,0.013205,0.001863,0.002478,0.000262,-0.019553,-0.058397,0.004397,5.014299e-07,-0.026568,-0.004843,-0.014207,0.001145,-0.000303,-0.002598,0.000371,-0.016562,-0.005212,-0.000287,-0.000302,-0.001630,-0.005985,-0.004350,-0.000248,-0.002451,0.006367,0.031117,-0.003621,0.064598,0.008057,0.032505,0.014850,0.010409,0.010662,0.008469,-0.013839,-0.001457,-0.028526,-0.004583,-0.011502,-0.011976,-0.046282,-0.001525,0.011740,1.246459e-02,-0.004729,-0.000319,0.002138,0.005743,0.003393
PC1,0.000818,0.002703,0.002483,0.001074,0.008414,0.001653,0.005403,-0.007031,-0.002352,-0.006668,0.011700,0.009130,-0.052199,0.004870,0.001386,-0.004829,-2.583812e-03,-0.006969,-0.001450,0.008231,0.001623,-0.004640,0.000863,0.003422,-0.007988,0.006610,0.003679,-0.000734,-0.005481,-0.002957,-0.006613,0.000755,0.000078,0.021060,-0.001337,-0.000506,0.000998,0.005391,0.007704,0.004377,0.004562,0.121059,0.020264,0.004472,0.002166,-0.005106,0.004421,0.003312,0.000723,0.000094,...,-0.006729,0.001444,-0.000505,0.004142,0.003387,0.000328,0.054160,0.000043,9.797513e-04,0.041357,0.000900,0.013345,-0.002079,-0.001149,-0.006444,0.002591,0.016265,0.013047,-0.001908,-0.000896,-0.002310,-0.001715,0.000616,0.003105,0.012036,0.001985,-0.002740,0.004625,-0.001177,0.000836,-0.000467,-0.007657,-0.001139,-0.001368,0.003402,0.011071,0.002570,0.018854,0.005728,0.006630,0.008994,-0.076998,-0.001730,0.000571,8.221552e-07,-0.012311,0.004439,0.002248,-0.000328,-0.000804
PC2,-0.002043,-0.004250,-0.006693,0.001043,0.000387,0.000684,-0.002190,0.004625,-0.007156,0.000145,-0.023411,-0.018006,-0.001754,-0.005379,-0.002349,0.000675,-1.474312e-04,-0.000394,-0.000089,-0.020530,-0.002400,0.000323,0.003756,-0.005201,0.006280,-0.006717,-0.003927,0.003946,0.015770,0.004650,0.048711,0.000349,0.000555,-0.010423,-0.002466,-0.000705,-0.006944,-0.014275,-0.011300,-0.003101,-0.002981,-0.034503,0.006135,0.015350,-0.000055,0.026171,0.016007,0.062153,-0.000281,0.001356,...,0.053734,-0.008655,0.004691,-0.005089,-0.002662,-0.009516,-0.041112,-0.003762,-2.731449e-03,-0.018427,-0.003115,-0.007391,-0.001173,-0.000505,-0.000499,-0.002683,-0.007173,0.006674,-0.000857,-0.001016,-0.000589,-0.002995,-0.003644,-0.002269,-0.005845,-0.011516,-0.020224,-0.004303,-0.021619,-0.005187,-0.006099,-0.007431,0.001968,-0.003719,-0.010594,-0.020671,-0.002634,-0.027143,0.000379,-0.005576,-0.004182,0.003497,-0.000524,-0.006837,-5.577974e-03,0.013115,-0.006562,-0.001171,0.005709,0.002006
PC3,0.003396,0.005048,0.004605,0.000487,-0.002822,-0.001382,-0.000449,-0.003691,0.000828,-0.003105,0.053840,0.023551,-0.030553,0.000614,0.000828,0.003287,-6.670267e-04,-0.0

## PCA DCM

This section inspects the PCA document-component matrix that places documents in the PCA component space.

In [7]:
PCA_DCM = inspect_table(
    section='PCA DCM',
    box_url='https://virginia.box.com/shared/static/uwq75d5c2d3e1dcvlrne11sjucuyctko',
    local_path='pca_dcm.parquet',
    file_format='parquet'
)

Section: PCA DCM
Box URL: https://virginia.box.com/shared/static/uwq75d5c2d3e1dcvlrne11sjucuyctko
Local file: pca_dcm.parquet
Loaded from: Box download
Format: parquet
Shape: 33,726 rows x 3,006 columns
Number of observations: 33,726
Column names:
PC0, PC1, PC2, PC3, PC4, PC5, PC6, PC7, PC8, PC9, PC10, PC11, PC12, PC13, PC14, PC15, PC16, PC17,
PC18, PC19, PC20, PC21, PC22, PC23, PC24, PC25, PC26, PC27, PC28, PC29, PC30, PC31, PC32, PC33,
PC34, PC35, PC36, PC37, PC38, PC39, PC40, PC41, PC42, PC43, PC44, PC45, PC46, PC47, PC48, PC49,
PC50, PC51, PC52, PC53, PC54, PC55, PC56, PC57, PC58, PC59, PC60, PC61, PC62, PC63, PC64, PC65,
PC66, PC67, PC68, PC69, PC70, PC71, PC72, PC73, PC74, PC75, PC76, PC77, PC78, PC79, PC80, PC81,
PC82, PC83, PC84, PC85, PC86, PC87, PC88, PC89, PC90, PC91, PC92, PC93, PC94, PC95, PC96, PC97,
PC98, PC99, PC100, PC101, PC102, PC103, PC104, PC105, PC106, PC107, PC108, PC109, PC110, PC111,
PC112, PC113, PC114, PC115, PC116, PC117, PC118, PC119, PC120, PC121, PC122, P

PC0       PC1       PC2       PC3       PC4  \
country_id  article_n                                                     
Afghanistan 1          0.210665 -0.027579 -0.056931  0.023060 -0.075607   
            2          0.028282 -0.002873  0.018675 -0.003923 -0.010452   
            3          0.081144  0.012865 -0.067011  0.025213 -0.045069   
            4          0.036248 -0.001025 -0.028926  0.009372 -0.012973   
            5          0.064454 -0.020695 -0.062457  0.002191 -0.065924   
            6          0.106634  0.009979  0.003753  0.048305 -0.050854   
            7          0.198942 -0.010473 -0.041032  0.032957 -0.034989   
            8          0.100001 -0.029839  0.017749 -0.005214 -0.015696   
            9          0.115856 -0.013223  0.018342  0.024399 -0.024211   
            10         0.138556  0.002136  0.007901  0.008888  0.046323   

                            PC5       PC6       PC7       PC8       PC9  \
country_id  article_n                                                     
Afghanistan 1         -0.099074 -0.023701  0.062940 -0.073178 -0.141306   
            2         -0.003854  0.049081  0.049474  0.020311  0.003358   
            3          0.032218  0.000346  0.018002  0.048850  0.023367   
            4          0.031950  0.017045  0.030386  0.023870  0.012059   
            5          0.028862  0.157077  0.003884 -0.019737  0.048431   
            6         -0.033930  0.099515  0.108876 -0.089064 -0.085167   
            7         -0.087302 -0.055771  0.011489 -0.010324 -0.103635   
            8          0.017471  0.015903  0.095749 -0.129115 -0.069369   
            9         -0.029182  0.046669  0.043469 -0.059914 -0.043894   
            10        -0.016826 -0.059387 -0.019701  0.017269 -0.001812   

                           PC10      PC11      PC12      PC13      PC14  \
country_id  article_n                                                     
Afghanistan 1          0.093413  0.013744  0.024961  0.000319  0.124925   
            2          0.033283 -0.033117 -0.026787  0.046269 -0.009705   
            3          0.053603 -0.086619 -0.037494  0.120289 -0.052639   
            4          0.033630 -0.048057 -0.033433  0.056139 -0.041449   
            5         -0.007002  0.149809 -0.015750 -0.007414  0.034788   
            6         -0.054336  0.042688  0.105186  0.005055  0.003456   
            7          0.051581  0.023584 -0.030923 -0.034464  0.074286   
            8          0.020443  0.041965 -0.078163 -0.056241  0.047418   
            9         -0.018288  0.050480  0.003171 -0.026829  0.016663   
            10        -0.033957  0.014352  0.011285 -0.096444 -0.008260   

                           PC15      PC16      PC17      PC18      PC19  \
country_id  article_n                                                     
Afghanistan 1         -0.006486  0.027834 -0.019398 -0.042186  0.047467   
            2         -0.015735 -0.032110 -0.022591 -0.018906 -0.012593   
            3          0.023280  0.067201  0.078033  0.025923  0.033825   
            4         -0.000750  0.019390  0.034420  0.003067  0.017207   
            5          0.131022 -0.041892 -0.042411 -0.065569  0.047492   
            6          0.013805  0.024862  0.007895 -0.036087  0.017812   
            7         -0.007688  0.017449 -0.029438 -0.035063 -0.014334   
            8         -0.064406  0.171728 -0.131972  0.027852 -0.014438   
            9          0.003008  0.025154 -0.021065 -0.013358  0.021443   
            10        -0.054962 -0.105535 -0.071720  0.052105  0.107470   

                           PC20      PC21      PC22      PC23      PC24  \
country_id  article_n                                                     
Afghanistan 1          0.010549 -0.075694 -0.021360  0.020613 -0.005342   
            2         -0.030071 -0.009179 -0.012634 -0.036277 -0.004964   
            3         -0.009032  0.020494  0.075700 -0.033219 -0.025731   
            4         -0.005436 -0.002266  0.0365

## PCA Loadings

This section inspects the PCA loadings table, which connects terms to principal components.

In [8]:
PCA_LOADINGS = inspect_table(
    section='PCA Loadings',
    box_url='https://virginia.box.com/shared/static/czlfhh4uprbcx2dg87jn8iyyft0yvaex',
    local_path='pca_loadings.parquet',
    file_format='parquet'
)

Section: PCA Loadings
Box URL: https://virginia.box.com/shared/static/czlfhh4uprbcx2dg87jn8iyyft0yvaex
Local file: pca_loadings.parquet
Loaded from: Box download
Format: parquet
Shape: 3,006 rows x 3,006 columns
Number of observations: 3,006
Column names:
PC0, PC1, PC2, PC3, PC4, PC5, PC6, PC7, PC8, PC9, PC10, PC11, PC12, PC13, PC14, PC15, PC16, PC17,
PC18, PC19, PC20, PC21, PC22, PC23, PC24, PC25, PC26, PC27, PC28, PC29, PC30, PC31, PC32, PC33,
PC34, PC35, PC36, PC37, PC38, PC39, PC40, PC41, PC42, PC43, PC44, PC45, PC46, PC47, PC48, PC49,
PC50, PC51, PC52, PC53, PC54, PC55, PC56, PC57, PC58, PC59, PC60, PC61, PC62, PC63, PC64, PC65,
PC66, PC67, PC68, PC69, PC70, PC71, PC72, PC73, PC74, PC75, PC76, PC77, PC78, PC79, PC80, PC81,
PC82, PC83, PC84, PC85, PC86, PC87, PC88, PC89, PC90, PC91, PC92, PC93, PC94, PC95, PC96, PC97,
PC98, PC99, PC100, PC101, PC102, PC103, PC104, PC105, PC106, PC107, PC108, PC109, PC110, PC111,
PC112, PC113, PC114, PC115, PC116, PC117, PC118, PC119, PC120, PC121, 

component_id,PC0,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,PC21,PC22,PC23,PC24,PC25,PC26,PC27,PC28,PC29,PC30,PC31,PC32,PC33,PC34,PC35,PC36,PC37,PC38,PC39,PC40,PC41,PC42,PC43,PC44,PC45,PC46,PC47,PC48,PC49,...,PC2956,PC2957,PC2958,PC2959,PC2960,PC2961,PC2962,PC2963,PC2964,PC2965,PC2966,PC2967,PC2968,PC2969,PC2970,PC2971,PC2972,PC2973,PC2974,PC2975,PC2976,PC2977,PC2978,PC2979,PC2980,PC2981,PC2982,PC2983,PC2984,PC2985,PC2986,PC2987,PC2988,PC2989,PC2990,PC2991,PC2992,PC2993,PC2994,PC2995,PC2996,PC2997,PC2998,PC2999,PC3000,PC3001,PC3002,PC3003,PC3004,PC3005
term_str,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
abide,0.004547,0.000818,-0.002043,0.003396,-0.004006,-0.001368,0.004920,0.006490,0.000539,-0.000997,0.005317,0.000324,0.006210,-0.002322,-0.001972,0.003677,0.001010,-0.000622,0.003572,-0.001445,0.000806,-0.002600,-0.001187,-0.001051,0.001336,-0.001671,-0.002661,-0.002383,-0.000702,5.110166e-04,-0.000166,-0.002350,-0.000373,0.000051,0.001838,0.002361,0.003661,-0.001456,-0.005429,-0.001650,0.000125,0.002013,0.002037,-0.002125,0.002263,-0.000864,-0.000269,0.003643,0.003663,-0.004539,...,0.002796,-0.000846,-0.001374,-0.003725,-0.000407,-0.000239,-0.001879,-0.000728,-0.000045,0.004036,-0.000262,0.001712,0.001194,0.000300,-0.000737,-0.000322,0.000102,-0.000824,0.000040,0.001265,-0.000750,-0.000251,-0.000882,0.000292,0.000378,-0.001077,-0.001767,-0.003012,0.000935,0.000599,0.000380,0.000454,0.000337,-0.000801,-0.000419,0.000614,0.000821,0.000506,-0.002007,0.000069,-0.000523,-0.000362,0.000063,0.000235,-0.000032,0.000094,0.000129,-0.000176,0.000034,-3.592310e-18
ability,0.004360,0.002703,-0.004250,0.005048,-0.004115,-0.008342,0.007147,0.005086,0.003806,-0.004755,0.011964,-0.000538,0.014553,-0.004220,-0.008341,0.005852,-0.000118,0.001400,0.003774,-0.004769,0.000469,-0.000170,0.001406,0.001984,-0.000892,0.002628,-0.003025,0.002934,-0.000230,-1.397544e-03,0.002921,0.001880,-0.001314,0.003040,0.001448,-0.001078,0.001966,-0.002091,-0.002280,0.001683,0.007469,-0.000853,0.003162,-0.004837,0.007170,-0.006114,0.000041,0.004688,0.006592,-0.015274,...,0.006871,-0.003967,0.000868,-0.023248,-0.002449,-0.003306,0.001483,-0.000674,0.003036,0.002517,0.003034,-0.002493,-0.002513,-0.008132,-0.000724,0.001314,-0.000679,0.005042,0.003138,-0.003915,0.000929,-0.001574,0.002094,0.000239,-0.002869,-0.004170,0.001961,-0.006797,0.000421,-0.002414,-0.022333,0.008561,-0.000053,0.000576,0.001332,-0.001283,0.001159,0.000044,-0.001005,0.000914,-0.004179,0.000274,0.001985,0.000514,0.000082,0.000233,-0.000658,0.000218,0.000153,8.607667e-17
able,-0.003622,0.002483,-0.006693,0.004605,0.000257,-0.007005,0.002142,-0.008452,-0.004266,-0.004264,-0.005445,-0.002982,-0.000193,-0.001651,0.004871,0.004160,-0.002122,-0.000543,0.005109,-0.004749,0.003161,-0.001918,0.000077,0.000828,-0.001927,-0.001312,-0.001032,0.001392,-0.000471,8.557377e-05,-0.005838,0.001321,-0.002672,0.000337,-0.000193,0.004609,-0.001161,-0.002207,-0.007545,-0.008578,0.004646,-0.007074,0.000648,-0.000712,0.002584,-0.003179,0.001436,0.003318,0.005450,0.004201,...,0.001902,-0.000036,0.001065,0.002172,-0.001851,-0.008063,0.002607,0.006083,0.002803,0.003414,0.000290,-0.001923,0.000518,0.000921,0.001426,0.000463,0.000771,-0.003004,0.003457,0.001103,0.000037,0.001370,-0.002295,-0.001499,0.000089,0.001533,0.000817,-0.000852,0.002113,0.002595,0.002711,-0.003520,-0.000372,-0.000967,0.002418,-0.003021,0.003196,0.000617,-0.001159,-0.001016,0.003047,-0.002059,-0.000130,-0.000355,0.000241,-0.000082,0.000102,0.000158,-0.000015,1.572708e-16
abolish,-0.000026,0.001074,0.001043,0.000487,0.001428,-0.002000,-0.000170,-0.000978,-0.003044,-0.000801,0.000572,0.000464,0.001020,0.000216,0.001030,0.000978,0.000585,0.000417,0.001163,0.000617,0.000076,0.001057,-0.001404,0.001515,0.001389,-0.000326,-0.001536,0.000743,-0.000167,-1.324527e-03,0.002198,-0.000682,-0.000642,0.001329,0.001343,-0.001171,-0.001282,-0.001172,0.001955,0.00391

## LDA DTM

This section inspects the LDA count matrix used as the source document-term table for topic modeling.

In [9]:
LDA_DTM = inspect_table(
    section='LDA DTM',
    box_url='https://virginia.box.com/shared/static/mfa6gtj63xpqedqjn2e4r394mx1y79ay',
    local_path='lda_dtm.parquet',
    file_format='parquet'
)

Section: LDA DTM
Box URL: https://virginia.box.com/shared/static/mfa6gtj63xpqedqjn2e4r394mx1y79ay
Local file: lda_dtm.parquet
Loaded from: Box download
Format: parquet
Shape: 33,220 rows x 3,504 columns
Number of observations: 33,220
Column names:
country_id, article_n, abandonment, abeyance, abide, abilities, ability, abolition, abroad,
abrogation, absence, absences, absent, absolute, abuse, abuses, accept, acceptance, accepts, access,
accessibility, accession, accident, accidents, accommodation, accomplices, accomplishment, accord,
accordance, accordingly, accords, account, accountability, accounting, accounts, accredit,
accreditation, accumulation, accusation, accusations, accused, achievement, achievements,
acknowledgement, acknowledgment, acquisition, acquittal, act, acting, action, actions, activities,
activity, acts, ad, adaptation, adaptations, addition, additions, address, addresses, adequate,
adherence, adjournment, adjudication, adjustment, adjustments, administer, administr

,country_id,article_n,abandonment,abeyance,abide,abilities,ability,abolition,abroad,abrogation,absence,absences,absent,absolute,abuse,abuses,accept,acceptance,accepts,access,accessibility,accession,accident,accidents,accommodation,accomplices,accomplishment,accord,accordance,accordingly,accords,account,accountability,accounting,accounts,accredit,accreditation,accumulation,accusation,accusations,accused,achievement,achievements,acknowledgement,acknowledgment,acquisition,acquittal,act,acting,action,...,weeks,weight,weights,welfare,wellbeing,whatsoever,whereof,whichever,whilst,white,wider,widow,widows,width,wife,wireless,wisdom,wish,wishes,withdrawal,withdrawals,withdrawn,withheld,withhold,withholding,witness,witnesses,woman,women,word,wording,words,work,worker,workers,working,workplace,works,world,worship,worth,writ,writing,writs,year,years,yellow,youth,zone,zones
0,Afghanistan,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Afghanistan,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Afghanistan,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Afghanistan,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,Afghanistan,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,Afghanistan,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,Afghanistan,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,Afghanistan,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,Afghanistan,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,Afghanistan,10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## LDA TOPIC

This section inspects the LDA topic summary table that lists the modeled topics and their top terms.

In [10]:
LDA_TOPIC = inspect_table(
    section='LDA TOPIC',
    box_url='https://virginia.box.com/shared/static/3tzt3hncq9z39786y93k4mvqw71sw6vt',
    local_path='lda_topics.parquet',
    file_format='parquet'
)

Section: LDA TOPIC
Box URL: https://virginia.box.com/shared/static/3tzt3hncq9z39786y93k4mvqw71sw6vt
Local file: lda_topics.parquet
Loaded from: Box download
Format: parquet
Shape: 20 rows x 3 columns
Number of observations: 20
Column names:
top_terms, doc_weight_mean, dominant_doc_count


,top_terms,doc_weight_mean,dominant_doc_count
topic_id,,,
T02,law laws courts matters jurisdiction decisions rules,0.079318,48
T13,office member election members term years person,0.073307,16
T15,rights people freedoms law citizens principles security,0.071653,29
T18,development education law resources services health protection,0.063536,24
T04,office person functions accordance advice appointment authority,0.060593,30
T00,days period law date time day force,0.057705,5
T06,duties functions law exercise members accordance powers,0.055221,3
T10,members majority vote votes election candidates number,0.054429,7
T17,government service power law authority powers legislation,0.053186,11


## LDA THETA

This section inspects the LDA `THETA` table containing document-by-topic weights.

In [11]:
LDA_THETA = inspect_table(
    section='LDA THETA',
    box_url='https://virginia.box.com/shared/static/oarntoraxn1mjwcuyiqdvvsvto4gv1si',
    local_path='lda_theta.parquet',
    file_format='parquet'
)

Section: LDA THETA
Box URL: https://virginia.box.com/shared/static/oarntoraxn1mjwcuyiqdvvsvto4gv1si
Local file: lda_theta.parquet
Loaded from: Box download
Format: parquet
Shape: 33,220 rows x 22 columns
Number of observations: 33,220
Column names:
country_id, article_n, T00, T01, T02, T03, T04, T05, T06, T07, T08, T09, T10, T11, T12, T13, T14,
T15, T16, T17, T18, T19


,country_id,article_n,T00,T01,T02,T03,T04,T05,T06,T07,T08,T09,T10,T11,T12,T13,T14,T15,T16,T17,T18,T19
0,Afghanistan,1,0.001042,0.001042,0.001042,0.062903,0.001042,0.001042,0.001042,0.001042,0.050439,0.001042,0.001042,0.001042,0.001042,0.001042,0.001042,0.868950,0.001042,0.001042,0.001042,0.001042
1,Afghanistan,2,0.016667,0.016667,0.016667,0.016667,0.016667,0.016667,0.016667,0.016667,0.683333,0.016667,0.016667,0.016667,0.016667,0.016667,0.016667,0.016667,0.016667,0.016667,0.016667,0.016667
2,Afghanistan,3,0.005556,0.005556,0.165357,0.222067,0.005556,0.122814,0.164047,0.005556,0.005556,0.005556,0.005556,0.242380,0.005556,0.005556,0.005556,0.005556,0.005556,0.005556,0.005556,0.005556
3,Afghanistan,4,0.012500,0.012500,0.012500,0.012500,0.012500,0.012500,0.012500,0.012500,0.012500,0.012500,0.012500,0.479471,0.012500,0.012500,0.012500,0.012500,0.295529,0.012500,0.012500,0.012500
4,Afghanistan,5,0.002778,0.002778,0.002778,0.002778,0.002778,0.002778,0.002778,0.255626,0.002778,0.002778,0.002778,0.002778,0.250175,0.002778,0.002778,0.446977,0.002778,0.002778,0.002778,0.002778
5,Afghanistan,6,0.004545,0.004545,0.004545,0.004545,0.004545,0.004545,0.265707,0.004545,0.130454,0.004545,0.004545,0.004545,0.004545,0.004545,0.004545,0.526566,0.004545,0.004545,0.004545,0.004545
6,Afghanistan,7,0.002941,0.002941,0.002941,0.002941,0.002941,0.002941,0.002941,0.002941,0.053735,0.002941,0.002941,0.002941,0.002941,0.002941,0.002941,0.741467,0.002941,0.002941,0.154798,0.002941
7,Afghanistan,8,0.005556,0.005556,0.005556,0.005556,0.005556,0.256323,0.005556,0.005556,0.240068,0.005556,0.005556,0.005556,0.005556,0.005556,0.005556,0.005556,0.005556,0.005556,0.409164,0.005556
8,Afghanistan,9,0.005556,0.005556,0.005556,0.005556,0.005556,0.005556,0.005556,0.005556,0.123324,0.005556,0.005556,0.005556,0.005556,0.005556,0.005556,0.776676,0.005556,0.005556,0.005556,0.005556
9,Afghanistan,10,0.006250,0.006250,0.006250,0.006250,0.006250,0.006250,0.006250,0.006250,0.116168,0.006250,0.006250,0.006250,0.006250,0.006250,0.006250,0.006250,0.006250,0.006250,0.771332,0.006250


## LDA PHI

This section inspects the LDA `PHI` table containing topic-by-term weights.

In [12]:
LDA_PHI = inspect_table(
    section='LDA PHI',
    box_url='https://virginia.box.com/shared/static/m8epjv3bddqkak7lt7qydwlziwq2euu5',
    local_path='lda_phi.parquet',
    file_format='parquet'
)

Section: LDA PHI
Box URL: https://virginia.box.com/shared/static/m8epjv3bddqkak7lt7qydwlziwq2euu5
Local file: lda_phi.parquet
Loaded from: Box download
Format: parquet
Shape: 20 rows x 3,502 columns
Number of observations: 20
Column names:
abandonment, abeyance, abide, abilities, ability, abolition, abroad, abrogation, absence, absences,
absent, absolute, abuse, abuses, accept, acceptance, accepts, access, accessibility, accession,
accident, accidents, accommodation, accomplices, accomplishment, accord, accordance, accordingly,
accords, account, accountability, accounting, accounts, accredit, accreditation, accumulation,
accusation, accusations, accused, achievement, achievements, acknowledgement, acknowledgment,
acquisition, acquittal, act, acting, action, actions, activities, activity, acts, ad, adaptation,
adaptations, addition, additions, address, addresses, adequate, adherence, adjournment,
adjudication, adjustment, adjustments, administer, administration, administrations, adminis

term_str,abandonment,abeyance,abide,abilities,ability,abolition,abroad,abrogation,absence,absences,absent,absolute,abuse,abuses,accept,acceptance,accepts,access,accessibility,accession,accident,accidents,accommodation,accomplices,accomplishment,accord,accordance,accordingly,accords,account,accountability,accounting,accounts,accredit,accreditation,accumulation,accusation,accusations,accused,achievement,achievements,acknowledgement,acknowledgment,acquisition,acquittal,act,acting,action,actions,activities,...,weeks,weight,weights,welfare,wellbeing,whatsoever,whereof,whichever,whilst,white,wider,widow,widows,width,wife,wireless,wisdom,wish,wishes,withdrawal,withdrawals,withdrawn,withheld,withhold,withholding,witness,witnesses,woman,women,word,wording,words,work,worker,workers,working,workplace,works,world,worship,worth,writ,writing,writs,year,years,yellow,youth,zone,zones
topic_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
T00,0.050000,0.05,0.050000,0.050000,0.050000,0.050000,0.050000,10.505440,0.050000,0.05,0.050000,10.971251,0.050000,0.050000,0.050000,0.050000,0.05,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.05,0.05,9.054885,309.643221,12.266023,0.050000,15.155393,0.050000,0.050000,0.050000,0.050000,0.05,0.050000,0.05,0.050000,0.05,0.05,0.05,1.281459,0.050000,0.050000,0.05,17.736116,0.050000,79.577504,0.050000,0.050000,...,106.727679,0.050000,0.050000,0.050000,0.050000,2.958228,7.949783,12.974700,0.050000,0.05,0.05000,0.050000,32.225263,0.050000,0.05,0.05,0.050000,4.412144,0.050000,0.050000,0.05,0.050000,17.05,11.05,2.269568,0.05000,0.050000,0.050000,0.050000,0.050000,13.835908,17.200677,0.050001,0.05,0.0500,0.050000,0.05,0.050000,4.158147,0.050000,0.050000,0.050000,30.086025,24.927260,401.591806,163.301817,0.05,0.050000,0.05,0.050000
T01,0.050000,0.05,0.050000,0.050000,0.050000,38.520938,0.050000,0.050000,3.663465,0.05,0.050000,0.050000,0.050000,0.050000,1.488825,0.275142,0.05,4.302121,0.050000,0.050000,0.050000,0.050000,0.050000,14.05,0.05,0.050000,238.038942,0.050000,1.805874,169.787925,0.050000,0.050011,334.473835,0.050000,0.05,0.050000,0.05,18.028042,0.05,0.05,0.05,0.050000,0.050000,0.050000,0.05,0.058109,0.050000,11.866961,0.050000,0.050000,...,0.115991,0.050000,0.050000,0.050000,2.260668,0.050000,0.050000,0.050000,0.050000,0.05,0.05000,25.762139,0.050000,0.050000,0.05,0.05,0.050000,0.050000,0.050000,141.713607,0.05,0.050000,0.05,1.05,13.909838,0.05000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.05,0.0500,0.050000,0.05,0.050000,0.050000,0.050000,3.175356,0.050000,10.910630,0.050000,1591.906510,4.314983,0.05,0.050000,0.05,0.050000
T02,0.050000,0.05,0.050000,0.050000,0.050000,27.202486,0.050000,0.050000,0.714187,0.05,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.05,0.050000,5.875918,0.050000,0.050000,1.732522,0.050000,0.05,0.05,86.605206,584.254725,3.021769,0.050106,5.057300,0.050000,6.606934,115.382191,0.050000,0.05,1.273555,0.05,0.050000,0.05,0.05,0.05,0.050000,0.050000,0.050000,0.05,67.342112,0.050000,63.691985,105.756718,71.465284,...,0.050000,0.050000,0.050000,0.050000,0.050000,6.472160,0.050000,0.050000,0.050000,0.05,0.05000,0.050000,0.050000,0.050000,0.05,0.05,0.050000,0.050000,0.050000,0.050000,0.05,0.050000,0.05,0.05,0.050000,0.05000,0.050000,0.050000,0.050000,0.050000,0.050000,0.303893,3.913876,0.05,0.0500,0.050000,0.05,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,25.709541,1.105993,3.374678,0.05,0.050000,0.05,4.718530
T03,0.050000,0.05,0.050000,0.050000,0.051118,0.050000,5.684324,0.050000,0.050000,0.05,0.050000,0.050000,11.345172,16.072002,0.050000,0.050000,0.05,321.030936,0.050000,0.050000,0.050000,0.050000,0.050000,0.05,0.05,0.050000,21.260599,0.050000,0.050000,59.381145,0.396668,4.476613,0.050000,0.050000,0.05,0.050000,0.05,0.050000,0.05,0.05,0.05,0.050000,0.050000,0.050000,0.05,9.221310,0.050000,82.596535,0.122263,27.552350,...,0.050000,0.050000,0.050026,0.050000,0.050000,12.877134,0.050000,0.050000,0.

## Sentiment VOCAB_SENT

This section inspects the sentiment-enriched vocabulary table produced by matching project vocabulary terms to the Loughran-McDonald lexicon.

In [13]:
VOCAB_SENT = inspect_table(
    section='Sentiment VOCAB_SENT',
    box_url='https://virginia.box.com/shared/static/9i317on9zav6qhx9o898dwxtwegjxqul',
    local_path='vocab_sent.parquet',
    file_format='parquet'
)

Section: Sentiment VOCAB_SENT
Box URL: https://virginia.box.com/shared/static/9i317on9zav6qhx9o898dwxtwegjxqul
Local file: vocab_sent.parquet
Loaded from: Box download
Format: parquet
Shape: 15,294 rows x 26 columns
Number of observations: 15,294
Column names:
term_str, n, p, i, df, idf, dfidf, n_chars, max_pos, max_pos_group, n_pos, cat_pos, n_pos_group,
cat_pos_group, stop, porter_stem, negative, positive, uncertainty, litigious, strong_modal,
weak_modal, constraining, complexity, sentiment, matched_lexicon


,term_str,n,p,i,df,idf,dfidf,n_chars,max_pos,max_pos_group,n_pos,cat_pos,n_pos_group,cat_pos_group,stop,porter_stem,negative,positive,uncertainty,litigious,strong_modal,weak_modal,constraining,complexity,sentiment,matched_lexicon
0,abandon,11,2.864780e-06,18.413144,11,5.007495,55.082440,7,VB,VB,2,"{JJ, VB}",2,"{JJ, VB}",0,abandon,1,0,0,0,0,0,0,0,-1,1
1,abandoned,35,9.115210e-06,16.743293,27,3.785102,102.197757,9,VBN,VB,8,"{IN, JJ, NN, NNP, NNPS, VB, VBD, VBN}",4,"{IN, JJ, NN, VB}",0,abandon,1,0,0,0,0,0,0,0,-1,1
2,abandoning,1,2.604346e-07,21.872576,1,7.592457,7.592457,10,VBG,VB,1,{VBG},1,{VB},0,abandon,1,0,0,0,0,0,0,0,-1,1
3,abandonment,40,1.041738e-05,16.550648,32,3.548063,113.538013,11,NN,NN,4,"{JJ, NN, NNP, VB}",3,"{JJ, NN, VB}",0,abandon,1,0,0,0,0,0,0,0,-1,1
4,abandons,4,1.041738e-06,19.872576,3,6.592457,19.777371,8,VBZ,VB,1,{VBZ},1,{VB},0,abandon,1,0,0,0,0,0,0,0,-1,1
5,abases,1,2.604346e-07,21.872576,1,7.592457,7.592457,6,VBZ,VB,1,{VBZ},1,{VB},0,abas,0,0,0,0,0,0,0,0,0,1
6,abate,4,1.041738e-06,19.872576,2,7.007495,14.014989,5,VB,VB,1,{VB},1,{VB},0,abat,0,0,0,0,0,0,0,0,0,1
7,abatement,2,5.208691e-07,20.872576,2,7.007495,14.014989,9,NN,NN,1,{NN},1,{NN},0,abat,0,0,0,0,0,0,0,0,0,1
8,abatements,2,5.208691e-07,20.872576,2,7.007495,14.014989,10,NNS,NN,1,{NNS},1,{NN},0,abat,0,0,0,0,0,0,0,0,0,1
9,abattoir,1,2.604346e-07,21.872576,1,7.592457,7.592457,8,NN,NN,1,{NN},1,{NN},0,abattoir,0,0,0,0,0,0,0,0,0,1


## Sentiment Source Lexicon

This section inspects the Loughran-McDonald source lexicon file that supports the sentiment notebook.

In [14]:
LM_LEXICON = inspect_table(
    section='Sentiment Source Lexicon',
    box_url='https://virginia.box.com/shared/static/obtdoavt4ue8cv0pi128b91gksayehnf',
    local_path='Loughran-McDonald_MasterDictionary_1993-2025.csv',
    file_format='csv'
)

Section: Sentiment Source Lexicon
Box URL: https://virginia.box.com/shared/static/obtdoavt4ue8cv0pi128b91gksayehnf
Local file: Loughran-McDonald_MasterDictionary_1993-2025.csv
Loaded from: Box download
Format: csv
Shape: 86,553 rows x 17 columns
Number of observations: 86,553
Column names:
Word, Seq_num, Word Count, Word Proportion, Average Proportion, Std Dev, Doc Count, Negative,
Positive, Uncertainty, Litigious, Strong_Modal, Weak_Modal, Constraining, Complexity, Syllables,
Source


,Word,Seq_num,Word Count,Word Proportion,Average Proportion,Std Dev,Doc Count,Negative,Positive,Uncertainty,Litigious,Strong_Modal,Weak_Modal,Constraining,Complexity,Syllables,Source
0,AARDVARK,1,814,3.085383e-08,2.022181e-08,4.059339e-06,158,0,0,0,0,0,0,0,0,2,12of12inf
1,AARDVARKS,2,3,1.137119e-10,7.898767e-12,8.829342e-09,1,0,0,0,0,0,0,0,0,2,12of12inf
2,ABACI,3,9,3.411357e-10,1.067549e-10,5.054031e-08,7,0,0,0,0,0,0,0,0,3,12of12inf
3,ABACK,4,29,1.099215e-09,6.073925e-10,1.523804e-07,28,0,0,0,0,0,0,0,0,2,12of12inf
4,ABACUS,5,9921,3.760453e-07,3.836730e-07,3.392121e-05,1355,0,0,0,0,0,0,0,0,3,12of12inf
5,ABACUSES,6,0,0.000000e+00,0.000000e+00,0.000000e+00,0,0,0,0,0,0,0,0,0,4,12of12inf
6,ABAFT,7,4,1.516159e-10,2.101878e-11,2.349506e-08,1,0,0,0,0,0,0,0,0,2,12of12inf
7,ABALONE,8,151,5.723499e-09,4.678978e-09,1.022283e-06,54,0,0,0,0,0,0,0,0,4,12of12inf
8,ABALONES,9,1,3.790397e-11,7.560853e-11,8.451617e-08,1,0,0,0,0,0,0,0,0,4,12of12inf
9,ABANDON,10,161792,6.132559e-06,4.843331e-06,3.244019e-05,79131,2009,0,0,0,0,0,0,0,3,12of12inf


## Sentiment BOW_SENT

This section inspects the bag-of-words table after sentiment dimensions from `VOCAB_SENT` have been mapped onto each bag-term row.

In [15]:
BOW_SENT = inspect_table(
    section='Sentiment BOW_SENT',
    box_url='https://virginia.box.com/shared/static/md0swqfp719tqns4f1m1015t3rqjcnk8',
    local_path='bow_sent.parquet',
    file_format='parquet'
)

Section: Sentiment BOW_SENT
Box URL: https://virginia.box.com/shared/static/md0swqfp719tqns4f1m1015t3rqjcnk8
Local file: bow_sent.parquet
Loaded from: Box download
Format: parquet
Shape: 1,307,209 rows x 23 columns
Number of observations: 1,307,209
Column names:
country_id, article_n, term_str, n, tfidf, negative, positive, uncertainty, litigious, strong_modal,
weak_modal, constraining, complexity, sentiment, negative_w, positive_w, uncertainty_w, litigious_w,
strong_modal_w, weak_modal_w, constraining_w, complexity_w, sentiment_w


,country_id,article_n,term_str,n,tfidf,negative,positive,uncertainty,litigious,strong_modal,weak_modal,constraining,complexity,sentiment,negative_w,positive_w,uncertainty_w,litigious_w,strong_modal_w,weak_modal_w,constraining_w,complexity_w,sentiment_w
0,Afghanistan,1,accordance,1,2.950516,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Afghanistan,1,adhering,1,9.123469,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Afghanistan,1,admiring,1,10.732907,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Afghanistan,1,all,3,8.100179,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,Afghanistan,1,almighty,1,6.597740,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,Afghanistan,1,appreciating,1,9.816616,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,Afghanistan,1,appropriate,1,4.486800,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,Afghanistan,1,approved,1,4.600594,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,Afghanistan,1,atrocity,1,10.732907,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,Afghanistan,1,attain,1,7.437070,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1


## Sentiment DOC_SENT

This section inspects the article-level sentiment summary table aggregated from `BOW_SENT`.

In [16]:
DOC_SENT = inspect_table(
    section='Sentiment DOC_SENT',
    box_url='https://virginia.box.com/shared/static/jg5stv77zs6b7fjnl5xrf34q0m8q33h9',
    local_path='doc_sent.parquet',
    file_format='parquet'
)

Section: Sentiment DOC_SENT
Box URL: https://virginia.box.com/shared/static/jg5stv77zs6b7fjnl5xrf34q0m8q33h9
Local file: doc_sent.parquet
Loaded from: Box download
Format: parquet
Shape: 33,725 rows x 40 columns
Number of observations: 33,725
Column names:
country_id, article_n, matched_token_count, matched_term_count, negative_w, positive_w,
uncertainty_w, litigious_w, strong_modal_w, weak_modal_w, constraining_w, complexity_w, sentiment_w,
negative_mean, positive_mean, uncertainty_mean, litigious_mean, strong_modal_mean, weak_modal_mean,
constraining_mean, complexity_mean, sentiment_mean, doc_id, year_created, year_amended, region,
region_compressed, source_file, file_format, char_count, line_count, article_count, clause_count,
token_count, v2x_regime, v2x_freexp_altinf, v2x_rule, v2x_regime_cat, v2x_freexp_altinf_cat,
v2x_rule_cat


,country_id,article_n,matched_token_count,matched_term_count,negative_w,positive_w,uncertainty_w,litigious_w,strong_modal_w,weak_modal_w,constraining_w,complexity_w,sentiment_w,negative_mean,positive_mean,uncertainty_mean,litigious_mean,strong_modal_mean,weak_modal_mean,constraining_mean,complexity_mean,sentiment_mean,doc_id,year_created,year_amended,region,region_compressed,source_file,file_format,char_count,line_count,article_count,clause_count,token_count,v2x_regime,v2x_freexp_altinf,v2x_rule,v2x_regime_cat,v2x_freexp_altinf_cat,v2x_rule_cat
0,Afghanistan,1,143,119,3,8,1,4,0,0,1,1,5,0.020979,0.055944,0.006993,0.027972,0.0,0.0,0.006993,0.006993,0.034965,Afghanistan_2004,2004,2004,Southern Asia,Asia,Afghanistan_2004.txt,plaintext,66806,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low
1,Afghanistan,2,5,5,0,0,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,Afghanistan_2004,2004,2004,Southern Asia,Asia,Afghanistan_2004.txt,plaintext,66806,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low
2,Afghanistan,3,15,14,0,0,0,1,0,0,0,0,0,0.000000,0.000000,0.000000,0.066667,0.0,0.0,0.000000,0.000000,0.000000,Afghanistan_2004,2004,2004,Southern Asia,Asia,Afghanistan_2004.txt,plaintext,66806,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low
3,Afghanistan,4,6,6,0,0,0,2,0,0,0,0,0,0.000000,0.000000,0.000000,0.333333,0.0,0.0,0.000000,0.000000,0.000000,Afghanistan_2004,2004,2004,Southern Asia,Asia,Afghanistan_2004.txt,plaintext,66806,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low
4,Afghanistan,5,35,30,1,0,0,2,0,0,0,0,-1,0.028571,0.000000,0.000000,0.057143,0.0,0.0,0.000000,0.000000,-0.028571,Afghanistan_2004,2004,2004,Southern Asia,Asia,Afghanistan_2004.txt,plaintext,66806,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low
5,Afghanistan,6,18,18,1,2,0,2,0,0,0,0,1,0.055556,0.111111,0.000000,0.111111,0.0,0.0,0.000000,0.000000,0.055556,Afghanistan_2004,2004,2004,Southern Asia,Asia,Afghanistan_2004.txt,plaintext,66806,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low
6,Afghanistan,7,31,29,0,2,0,1,0,0,1,0,2,0.000000,0.064516,0.000000,0.032258,0.0,0.0,0.032258,0.000000,0.064516,Afghanistan_2004,2004,2004,Southern Asia,Asia,Afghanistan_2004.txt,plaintext,66806,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low
7,Afghanistan,8,27,26,0,0,0,0,0,0,1,1,0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.037037,0.037037,0.000000,Afghanistan_2004,2004,2004,Southern Asia,Asia,Afghanistan_2004.txt,plaintext,66806,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low
8,Afghanistan,9,19,19,0,2,0,1,0,0,0,0,2,0.000000,0.105263,0.000000,0.052632,0.0,0.0,0.000000,0.000000,0.105263,Afghanistan_2004,2004,2004,Southern Asia,Asia,Afghanistan_2004.txt,plaintext,66806,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low
9,Afghanistan,10,20,18,0,0,0,2,0,0,0,0,0,0.000000,0.000000,0.000000,0.100000,0.0,0.0,0.000000,0.000000,0.000000,Afghanistan_2004,2004,2004,Southern Asia,Asia,Afghanistan_2004.txt,plaintext,66806,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low


## VOCAB_W2V

This section inspects the Word2Vec vocabulary table that adds embedding features to the project `VOCAB` table.

In [17]:
VOCAB_W2V = inspect_table(
    section='VOCAB_W2V',
    box_url='https://virginia.box.com/shared/static/g397pdb178mn0efv68s8p4tlpgjsybc4',
    local_path='vocab_w2v.parquet',
    file_format='parquet'
)

Section: VOCAB_W2V
Box URL: https://virginia.box.com/shared/static/g397pdb178mn0efv68s8p4tlpgjsybc4
Local file: vocab_w2v.parquet
Loaded from: Box download
Format: parquet
Shape: 5,622 rows x 272 columns
Number of observations: 5,622
Column names:
term_str, n, p, i, df, idf, dfidf, n_chars, max_pos, max_pos_group, n_pos, cat_pos, n_pos_group,
cat_pos_group, stop, porter_stem, w2v_0, w2v_1, w2v_2, w2v_3, w2v_4, w2v_5, w2v_6, w2v_7, w2v_8,
w2v_9, w2v_10, w2v_11, w2v_12, w2v_13, w2v_14, w2v_15, w2v_16, w2v_17, w2v_18, w2v_19, w2v_20,
w2v_21, w2v_22, w2v_23, w2v_24, w2v_25, w2v_26, w2v_27, w2v_28, w2v_29, w2v_30, w2v_31, w2v_32,
w2v_33, w2v_34, w2v_35, w2v_36, w2v_37, w2v_38, w2v_39, w2v_40, w2v_41, w2v_42, w2v_43, w2v_44,
w2v_45, w2v_46, w2v_47, w2v_48, w2v_49, w2v_50, w2v_51, w2v_52, w2v_53, w2v_54, w2v_55, w2v_56,
w2v_57, w2v_58, w2v_59, w2v_60, w2v_61, w2v_62, w2v_63, w2v_64, w2v_65, w2v_66, w2v_67, w2v_68,
w2v_69, w2v_70, w2v_71, w2v_72, w2v_73, w2v_74, w2v_75, w2v_76, w2v_77, w2v_78,

,term_str,n,p,i,df,idf,dfidf,n_chars,max_pos,max_pos_group,n_pos,cat_pos,n_pos_group,cat_pos_group,stop,porter_stem,w2v_0,w2v_1,w2v_2,w2v_3,w2v_4,w2v_5,w2v_6,w2v_7,w2v_8,w2v_9,w2v_10,w2v_11,w2v_12,w2v_13,w2v_14,w2v_15,w2v_16,w2v_17,w2v_18,w2v_19,w2v_20,w2v_21,w2v_22,w2v_23,w2v_24,w2v_25,w2v_26,w2v_27,w2v_28,w2v_29,w2v_30,w2v_31,w2v_32,w2v_33,...,w2v_206,w2v_207,w2v_208,w2v_209,w2v_210,w2v_211,w2v_212,w2v_213,w2v_214,w2v_215,w2v_216,w2v_217,w2v_218,w2v_219,w2v_220,w2v_221,w2v_222,w2v_223,w2v_224,w2v_225,w2v_226,w2v_227,w2v_228,w2v_229,w2v_230,w2v_231,w2v_232,w2v_233,w2v_234,w2v_235,w2v_236,w2v_237,w2v_238,w2v_239,w2v_240,w2v_241,w2v_242,w2v_243,w2v_244,w2v_245,w2v_246,w2v_247,w2v_248,w2v_249,w2v_250,w2v_251,w2v_252,w2v_253,w2v_254,w2v_255
0,abandoned,35,0.000009,16.743293,27,3.785102,102.197757,9,VBN,VB,8,"{IN, JJ, NN, NNP, NNPS, VB, VBD, VBN}",4,"{IN, JJ, NN, VB}",0,abandon,0.073700,-0.174708,0.093871,-0.000434,0.021934,0.122396,-0.112501,0.088526,-0.084787,0.155957,-0.043394,0.042839,-0.122988,-0.026982,-0.001589,-0.139062,-0.087802,0.163906,0.021254,-0.085140,0.122532,-0.217843,0.039072,-0.132595,0.063915,0.012649,-0.058430,0.069377,-0.011684,-0.056713,-0.066475,-0.031281,0.096206,0.141939,...,0.139267,0.012704,-0.154450,0.197402,-0.048538,-0.006751,-0.100635,-0.045073,-0.106387,-0.055860,0.072501,-0.093267,-0.043283,-0.061011,-0.074880,0.059546,0.117177,-0.064828,-0.087773,0.019939,0.012150,-0.089868,-0.043985,-0.025825,0.094725,0.031734,-0.072790,0.030077,-0.053025,-0.191701,0.089533,0.008379,-0.076043,-0.200846,-0.199409,0.005384,-0.078783,-0.018475,0.097394,-0.232650,-0.043581,0.120082,0.100029,-0.088394,-0.032106,-0.269811,-0.000878,-0.131548,-0.046129,-0.001895
1,abandonment,40,0.000010,16.550648,32,3.548063,113.538013,11,NN,NN,4,"{JJ, NN, NNP, VB}",3,"{JJ, NN, VB}",0,abandon,0.088658,-0.175913,0.147324,0.099200,0.015570,0.158780,-0.006007,-0.044386,-0.093620,0.065705,-0.055803,0.131556,0.101062,0.088443,-0.031846,-0.116678,-0.009710,0.145142,0.134152,-0.068435,0.068232,-0.133187,-0.021322,0.006533,0.057720,0.140106,-0.117492,0.079170,0.045317,-0.031481,0.027261,0.045549,0.004103,0.098799,...,-0.034857,-0.034180,-0.106529,0.013614,-0.002248,-0.043029,-0.177203,-0.038882,-0.128045,0.048825,0.010827,-0.101246,0.012953,-0.047898,-0.012416,-0.016143,0.096819,-0.030208,-0.092434,-0.019137,0.008156,0.013714,-0.023656,0.043786,0.124049,-0.056937,0.082849,0.008655,0.068120,0.031792,0.115240,0.074317,-0.036700,-0.138439,-0.032085,0.086659,-0.134960,0.042748,-0.023416,-0.126393,-0.048050,-0.057721,0.138578,-0.042252,0.034308,-0.145910,0.040471,-0.075106,-0.105863,-0.140268
2,abeyance,35,0.000009,16.743293,5,6.007495,30.037473,8,NN,NN,3,"{CC, JJ, NN}",3,"{CC, JJ, NN}",0,abey,0.281123,-0.275636,-0.104653,-0.091375,-0.083771,0.034206,-0.138228,-0.068119,0.046710,0.061548,0.055645,0.115365,0.073761,-0.329504,0.215103,-0.000641,-0.247402,-0.020216,-0.178775,0.143593,-0.142429,0.001881,-0.254678,-0.028050,0.121308,-0.390504,0.048189,0.251398,0.213564,0.105306,0.224514,-0.174981,0.149835,0.222569,...,0.217903,0.143233,0.101785,0.422052,0.217841,-0.172943,0.104746,0.137911,-0.167356,-0.116892,-0.007192,-0.086031,-0.036458,-0.119800,-0.077488,0.234456,0.039391,0.151356,-0.102520,0.037541,-0.155224,0.033608,-0.264599,-0.031395,0.101758,0.071054,-0.026766,0.016328,0.001893,-0.050023,-0.087428,0.159665,0.159006,-0.058171,-0.307542,-0.025018,0.119891,-0.128965,-0.076891,-0.366557,0.107452,0.071837,0.011519,-0.095819,-0.062295,-0.323261,-0.018861,0.007049,-0.196832,-0.059989
3,abide,104,0.000027,15.172136,51,2.892017,147.492883,5,VB,VB,4,"{NN, RB, VB, VBP}",3,"{NN, RB, VB}",0,abid,0.281837,-0.215503,0.067570,-0.199556,0.199987,0.154022,0.796452,-0.164625,-0.078652,0.224606,-0.258865,-0.105670,0.323694,-0.026433,-0.192868,-0.085982,0.084579,0.056524,0.332158,0.289530,0.022649,-0.284675,-0.111188,-0.103307,0.180317,-0.258130,-0.045990,-0.032062,-0.242827,-0.234238,-0.103869,0.037107,-0.150219,0.026284,...,-0.068272,-0.136070,0.153113,-0.416895,